# Protein-ligand binding free energy calculations 

Xiping Gong (xipinggong@uga.edu) from the [Jack Huang's Lab](https://site.caes.uga.edu/huanglab/)


# Introduction

Accurately estimating the binding free energy between a protein and a ligand is a crucial step in computational drug design and environmental toxicology studies. Molecular mechanics-based free energy calculations, such as Molecular Mechanics Poisson–Boltzmann Surface Area (MMPBSA) and Molecular Mechanics Generalized Born Surface Area (MMGBSA), provide a computationally efficient alternative to more expensive methods like Free Energy Perturbation (FEP) and Thermodynamic Integration (TI). While FEP and TI offer high accuracy by explicitly modeling the thermodynamic cycle of ligand binding, they require extensive sampling and computational resources. In contrast, MMPBSA and MMGBSA approximate binding free energies using an ensemble of molecular dynamics (MD) snapshots, significantly reducing computational cost while maintaining reasonable accuracy for ranking ligand affinities.

The [gmx_MMPBSA tool](https://pubs.acs.org/doi/full/10.1021/acs.jctc.1c00645) has enabled seamless integration of MMPBSA calculations into the GROMACS molecular dynamics platform, making it accessible for researchers studying protein-ligand interactions. This tool automates the extraction of trajectory frames and energy calculations, allowing users to efficiently estimate binding free energies within the GROMACS ecosystem.

In our previous study, we employed AlphaFold 3 and AutoDock Vina for molecular docking to predict the binding poses of per- and polyfluoroalkyl substances (PFAS) with target proteins (Link: https://xipinggong.com/files/tutorials/pfas_docking.html). While docking provides a rapid assessment of binding configurations, it does not account for entropic and solvation effects, which are critical for determining binding affinities. Therefore, we applied the MMPBSA method to further refine our binding free energy estimates and assess its suitability for studying protein-PFAS interactions.

In this tutorial, we present a step-by-step guide on how to use the gmx_MMPBSA tool to calculate protein-ligand binding free energies. We begin with an overview of the MMPBSA methodology, followed by a practical guide on running gmx_MMPBSA from a downloaded protein-ligand complex PDB file. This workflow serves as a resource for researchers interested in applying MMPBSA calculations to evaluate protein-ligand interactions efficiently within the GROMACS framework.



# Methodology

Here, we used the Single Trajectory (ST) protocol for MMPBSA calculations, extracting protein, ligand, and complex energies from the same MD trajectory to improve efficiency and consistency. However, the ST protocol has limitations, including restricted conformational sampling of both unbound and bound states. If your systems are disordered, then this ST protocol can be inaccurate. Normally, the implicit solvent models are often used to accelerate calculations while capturing key solvation effects, and gmx_MMPBSA allows decomposition into van der Waals, electrostatic, polar, and non-polar solvation contributions. However, one key component of implicit solvent models is to estimate the solvation free energy, where some PB and GB models are often used, but they are often relatively accurate. Despite these challenges, it remains a computationally efficient method for estimating protein-ligand binding free energy, making it valuable for large-scale screening and comparative studies. So, it is essential to do the assessment.

Given a configuration of protein-ligand system (e.g., $X$), we can introduce a reduced potential energy $W(X) = U_{p-p}(X) + G_{solv}(X)$, to estimate its free energy, where the $U_{p-p}(X)$ is the potential energy of solute system (like a protein-ligand system), and $G_{solv}(X)$ captures the solvent effects on the solute system. The $U_{p-p}(X)$ can be readily captured by the conventional mechanical force fields of protein-ligand systems, while the $G_{solv}(X)$ is often captured by the PB or GB implicit solvent models. In the ST protocol, two configurations are considered, including the unbound and bound configuration, where their only difference is whether protein and ligand systems are separated or not.

An illustrated description can be found in the following image:  
![Illustration of gmx_MMPBSA](https://pubs.acs.org/cms/10.1021/acs.jctc.1c00645/asset/images/medium/ct1c00645_0001.gif)  
*Source: [ACS JCTC](https://pubs.acs.org/doi/10.1021/acs.jctc.1c00645)*

where, the $\Delta G_{bind}$ can be represented by the $W(X_{\text{bound}}) - W(X_{\text{unbound}})$.
Since the Single Trajectory (ST) protocol is used, intra-molecular interactions within both the protein and ligand are ignored. Instead, only inter-molecular interactions between the protein and ligand are considered, which can be further categorized into non-polar interactions (e.g., van der Waals forces) and polar interactions (e.g., electrostatic interactions). We can have the following energy decomposition.

$\Delta G_{bind} = \Delta U_{\text{gas}} + \Delta G_{\text{solv}} $
$                = \Delta U_{\text{gas}}^{\text{vdW}} + \Delta U_{\text{gas}}^{\text{ele}} + $
$                  \Delta G_{\text{solv}}^{\text{np}} + \Delta G_{\text{solv}}^{\text{polar}} $ (**Eq. 1**),

where the non-polar solvation free energy $\Delta G_{\text{solv}}^{\text{np}}$ is typically estimated using the Solvent Accessible Surface Area (SASA) model, assuming proportionality to the molecule's total SASA. In contrast, the polar solvation free energy $\Delta G_{\text{solv}}^{\text{polar}}$ is commonly calculated using the Poisson–Boltzmann (PB) or Generalized Born (GB) model, which involves more complex computations.


# Tutorial

In this section, I will show how we can use the gmx_MMPBSA tool to calculate the binding free energy of protein-ligand from a pdb structure. For example, this pdb structure only has one protein and one ligand. Several steps can be generally used for the gmx_MMPBSA calculations. 

## Prepare the pdb structure
```bash
# Go to the PDB website: https://www.rcsb.org, and then input the PDBID "7FEU", you will see what it looks like, which has one protein and one ligand.
# We can also directly download the PDB file
$ wget https://files.rcsb.org/download/7FEU.pdb

# Now, we will use the python scripts to process this downloaded pdb file.
# The scripts can be found from this link: https://github.com/XipingGong/pfas_docking.
# Here, I assume that the pfas_docking folder has been downloaded in your local computer.
$ python ~/program/pfas_docking/scripts/check_pdb.py 7FEU.pdb 
# It will show that it has one protein and 3 ligands, including the 4I6, HOH, and P6G.
# What we need is to save the protein and 4I6 ligand.
# The clean_pdb.py will check the missing heavy atoms and saved the cleaned pdb: 
# "cleaned_7FEU.pdb"
$ python ~/program/pfas_docking/scripts/clean_pdb.py 7FEU.pdb 
# Using the extract_pdb.py to save the protein-ligand system into a pdb file:
# cleaned_7FEU_4I6_133_Protein_ChainA.pdb
$ python ~/program/pfas_docking/scripts/extract_pdb.py cleaned_7FEU.pdb 4I6
# Copy it into an input pdb file named as "complex.pdb"
$ cp cleaned_7FEU_4I6_133_Protein_ChainA.pdb complex.pdb
```


### Directly generate the input for the gmx_MMPBSA
```bash
# It is necessary to have a look at an example of gmx_MMPBSA.
# Please see here: https://valdes-tresanco-ms.github.io/gmx_MMPBSA/dev/examples/Protein_ligand/ST
# This example requires several inputs, including
# mmpbsa.in: the input file of gmx_MMPBSA, which defines some parameters
# com.tpr: this is tpr file from the GROMACS, which includes all possible info, including the force field, topology, etc.
# com_traj.xtc: this is the trajectory file, which includes the coordinates of many trajectory frames. We assume that these 
#               trajectories are independently sampled and represent the major state of the protein-ligand complex.
# index.ndx: this is the index file created from GROMACS, which includes the indices of protein and ligand
#            "-cg 3 4" should refer to the protein and ligand in the index.ndx file.
# topol.top: this is the topol.top file created from GROMACS, which includes all parameters, etc. 
# However, personally, I think these info can be extracted from the tpr file, so I am not sure if we have to need this file, 
# but, so far, the gmx_MMPBSA needs it.
#
# In this way, we still need to use the GROMACS to obtain these input files for the gmx_MMPBSA calculations. 
# To run this, we need to install the gmxMMPBSA, which includes the installation of GROMACS.
# We also can have a look at another great tutorial on how to run the MD simulations.
# Link: http://www.mdtutorials.com/gmx/complex/
# After that, we first need to prepare the molecular topology and force field parameters for both protein and ligand.
# 
$ python ~/program/pfas_docking/scripts/get_inputs_for_vina.py complex.pdb # obtain the individual protein and ligand
$ mv x_receptor.pdb protein.pdb
$ mv x_ligand.pdb ligand.pdb
#
# Processing protein
$ gmx pdb2gmx -f protein.pdb -o protein.gro -p topol_protein.top -ff charmm27 -water tip3p
# -->
# $ gmx pdb2gmx -f protein.pdb -o protein.gro -p topol_protein.top # it will generate the topology file for protein
# 8: CHARMM27 all-atom force field (CHARM22 plus CMAP for proteins)
# 1: TIP3P   TIP 3-point, recommended
#
# Processing ligand
$ acpype -i ligand.pdb -c bcc -n -1 # we are using acpype to run, which will generate a "ligand.acpype" folder
# in this "ligand.acpype" folder, it has many output files, such as
# ligand.acpype/ligand_GMX.gro
# ligand.acpype/ligand_GMX.top
# ligand.acpype/ligand_GMX.itp
$ cp ligand.acpype/ligand_GMX.gro   ligand_GMX.gro
$ cp ligand.acpype/posre_ligand.itp posre_ligand.itp
$ cp ligand.acpype/ligand_GMX.top   ligand_GMX.top
$ cp ligand.acpype/ligand_GMX.itp   ligand_GMX.itp
# We then need to incorporate the topology file of ligand into the protein one, to get the final topol.top file.
# This step could take time to run.

# Merged into a complex system
$ python merge_gro.py protein.gro  ligand_GMX.gro > complex.gro # Manually generate the complex.gro file
$ gmx editconf -f complex.gro -o newbox.gro -bt cubic -d 1.0
$ python merge_top.py topol_protein.top ligand_GMX.top > topol.top

# Vaccum Energy minimization
$ # wget http://www.mdtutorials.com/gmx/complex/Files/em.mdp
$ gmx grompp -f em.mdp -c complex.gro -p topol.top -o em.tpr
$ gmx mdrun -v -deffnm em

# Save into tpr and trr files
$ cp em.tpr com.tpr
$ cp em.trr com_traj.trr

# Creating the index.ndx
echo -e "
name 1 receptor
name 13 ligand
1 | 13
! \"receptor_ligand\"
q
" | gmx make_ndx -f em.tpr -o index.ndx

```



### Explicit solvent simulations to generate the input for the gmx_MMPBSA
```bash
# It is necessary to have a look at an example of gmx_MMPBSA.
# Please see here: https://valdes-tresanco-ms.github.io/gmx_MMPBSA/dev/examples/Protein_ligand/ST
# This example requires several inputs, including
# mmpbsa.in: the input file of gmx_MMPBSA, which defines some parameters
# com.tpr: this is tpr file from the GROMACS, which includes all possible info, including the force field, topology, etc.
# com_traj.xtc: this is the trajectory file, which includes the coordinates of many trajectory frames. We assume that these 
#               trajectories are independently sampled and represent the major state of the protein-ligand complex.
# index.ndx: this is the index file created from GROMACS, which includes the indices of protein and ligand
#            "-cg 3 4" should refer to the protein and ligand in the index.ndx file.
# topol.top: this is the topol.top file created from GROMACS, which includes all parameters, etc. 
# However, personally, I think these info can be extracted from the tpr file, so I am not sure if we have to need this file, 
# but, so far, the gmx_MMPBSA needs it.
#
# In this way, we still need to use the GROMACS to obtain these input files for the gmx_MMPBSA calculations. 
# To run this, we need to install the gmxMMPBSA, which includes the installation of GROMACS.
# We also can have a look at another great tutorial on how to run the MD simulations.
# Link: http://www.mdtutorials.com/gmx/complex/
# After that, we first need to prepare the molecular topology and force field parameters for both protein and ligand.
# 
$ python ~/program/pfas_docking/scripts/get_inputs_for_vina.py complex.pdb # obtain the individual protein and ligand
$ mv x_receptor.pdb protein.pdb
$ mv x_ligand.pdb ligand.pdb
#
# Processing protein
$ gmx pdb2gmx -f protein.pdb -o protein.gro -p topol_protein.top -ff charmm27 -water tip3p
# -->
# $ gmx pdb2gmx -f protein.pdb -o protein.gro -p topol_protein.top # it will generate the topology file for protein
# 8: CHARMM27 all-atom force field (CHARM22 plus CMAP for proteins)
# 1: TIP3P   TIP 3-point, recommended
#
# Processing ligand
$ acpype -i ligand.pdb -c bcc -n -1 # we are using acpype to run, which will generate a "ligand.acpype" folder
# in this "ligand.acpype" folder, it has many output files, such as
# ligand.acpype/ligand_GMX.gro
# ligand.acpype/ligand_GMX.top
# ligand.acpype/ligand_GMX.itp
$ cp ligand.acpype/ligand_GMX.gro   ligand_GMX.gro
$ cp ligand.acpype/posre_ligand.itp posre_ligand.itp
$ cp ligand.acpype/ligand_GMX.top   ligand_GMX.top
$ cp ligand.acpype/ligand_GMX.itp   ligand_GMX.itp
# We then need to incorporate the topology file of ligand into the protein one, to get the final topol.top file.
# This step could take time to run.

# Merged into a complex system
$ python merge_gro.py protein.gro  ligand_GMX.gro > complex.gro # Manually generate the complex.gro file
$ gmx editconf -f complex.gro -o newbox.gro -bt cubic -d 1.0
$ python merge_top.py topol_protein.top ligand_GMX.top > topol.top

# Adding solvent & salts for the explicit solvent simulations
$ gmx solvate -cp newbox.gro -cs spc216.gro -p topol.top -o solv.gro
$ # wget http://www.mdtutorials.com/gmx/complex/Files/ions.mdp
$ gmx grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr
$ SOL_GROUP=$(gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral 2>&1 | grep -E "SOL\)" | awk '{print $2}')
$ echo $SOL_GROUP | gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral

# Creating the index.ndx
echo -e "
name 1 receptor
name 13 ligand
1 | 13
! \"receptor_ligand\"
q
" | gmx make_ndx -f em.tpr -o index.ndx

# Energy minimization
$ # wget http://www.mdtutorials.com/gmx/complex/Files/em.mdp
$ gmx grompp -f em.mdp -c solv_ions.gro -p topol.top -o em.tpr
$ gmx mdrun -v -deffnm em

# NVT Thermostats
$ # wget http://www.mdtutorials.com/gmx/complex/Files/nvt.mdp # modify nvt.mdp
$ gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -n index.ndx -o nvt.tpr
$ gmx mdrun -deffnm nvt -v

# NPT simulation
$ # wget http://www.mdtutorials.com/gmx/complex/Files/npt.mdp # modify npt.mdp
$ gmx grompp -f npt.mdp -c nvt.gro -t nvt.cpt -r nvt.gro -p topol.top -n index.ndx -o npt.tpr --maxwarn 1
$ gmx mdrun -deffnm npt -v

# MD production
$ # wget http://www.mdtutorials.com/gmx/complex/Files/md.mdp # modify md.mdp
$ gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -n index.ndx -o md.tpr
$ gmx mdrun -deffnm md -v

# Save into tpr and trr files
$ cp md.tpr com.tpr
$ cp md.xtc com_traj.xtc

```




## Run the ST protocol for protein-ligand complex using gmx_MMPBSA

```bash
$ rm -f FINAL_RESULTS_MMPBSA.dat # To remove the dat file if any
$ rm -f FINAL_RESULTS_MMPBSA.csv # To remove the dat file if any
$ gmx_MMPBSA -O -i mmpbsa.in -cs com.tpr -ct com_traj.xtc -ci index.ndx -cg 1 13 -cp topol.top -nogui -o FINAL_RESULTS_MMPBSA.dat -eo FINAL_RESULTS_MMPBSA.csv
$ cat FINAL_RESULTS_MMPBSA.dat

```


# Data analysis

In this section, I will show how to look at the data we collected.
First, we will look at the change of RMSD values for the trjectories collected, compared to the input pdb file. Then, we will check the free energies obtained from the gmx_MMPBSA.

## RMSD calculations
```bash
$ python calculate_rmsd.py em.gro md.xtc --ligandID 4I6
>> Frame   Protein backboone RMSD (nm)       Ligand RMSD (nm)        Complex RMSD (nm)
>> 0       0.028177        0.052365        0.036691
>> 1       0.072694        0.034999        0.072771
>> 2       0.079245        0.032625        0.078637
>> 3       0.073669        0.027221        0.072312
>> 4       0.083549        0.072444        0.084557
>> 5       0.095029        0.067737        0.095174
>> 6       0.083648        0.069730        0.086117
>> 7       0.080249        0.047186        0.081107
>> 8       0.085807        0.020840        0.084613
>> 9       0.085750        0.037580        0.086193
>> 10      0.099241        0.030270        0.097970
# It shows the RMSD values of protein backbone, ligand, and their protein-ligand heavy atoms.

```

# Appendix

```bash

$ mkdir 3RZ7_RZ7
$ bash gen_ligand_pdbs.sh /home/xg69107/work/pfas_pdbs/dock_dir/3RZ7_RZ7
$ bash gen_ligand_top.sh native_ligand.pdb

```